In [ ]:
import { RecursiveSet, Tuple, Value } from 'recursive-Set';
type Variable = string;
type Literal  = Variable 
              | Tuple<['¬', Variable]>;

In [ ]:
type RS<T extends Value> = RecursiveSet<T>;
type Clause = RS<Literal>;

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

The function $\texttt{complement}(l)$ computes the *complement* of a literal $l$.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either 
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [ ]:
function complement(l: Literal): Literal {
    if (typeof l == 'string') {
        return new Tuple('¬', l);
    } else {
        return l.get(1);
    }
}

In [ ]:
complement('p');

In [ ]:
complement(new Tuple('¬', 'p'));

The function $\texttt{extractVariable}(l)$ extracts the propositional variable from the literal $l$.
If $p$ is a propositional variable, we have the following: 

1. $\texttt{extractVariable}(p) = p$,
2. $\texttt{extractVariable}(\neg p) = p$.

In [ ]:
function extractVariable(l: Literal): Variable {
    if (typeof l == 'string') {
        return l;
    } else {
        return l.get(1);
    }
}

In [ ]:
extractVariable('p');

In [ ]:
extractVariable(new Tuple('¬', 'p'));

`empty(x)` creates the set $\{\}$.

In [ ]:
function empty<T extends Value>(): RS<T> {
    return new RecursiveSet<T>()
}

In [ ]:
empty<Variable>()

`single(x)` creates the set $\{\texttt{x}\}$.

In [ ]:
function single<T extends Value>(x: T): RS<T> {
    return new RecursiveSet<T>(x)
}

In [ ]:
single('x')

The function $\texttt{collectsVariables}(M)$ takes a set of clauses $M$ as its input and computes the set of all propositional variables occurring in $M$.  The clauses in $M$ are represented as sets of literals.

In [ ]:
function collectVariables(M: RS<Clause>): RS<Variable> {
    const r = empty<Variable>();
    return M.reduce((a, c) => a.union(c.map(extractVariable)), r);
}

In [ ]:
const C1: Clause = new RecursiveSet<Literal>('p', 'q', 'r');
const C2: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), new Tuple('¬', 'q'), new Tuple('¬', 's'));
const M: RecursiveSet<Clause> = new RecursiveSet<Clause>(C1, C2);
console.log(collectVariables(M).toString());

The function `prettifyClause(C)` converts the clause `C` into a string. 

In [ ]:
function prettifyClause(clause: RS<Literal>): string {
    const literals: string[] = [];
    for (const lit of clause) {
        if (typeof lit == 'string') {
            literals.push(lit);
        } else {
            literals.push(`${lit.get(0)}${lit.get(1)}`);
        }
    }
    return `{${literals.join(', ')}}`;
}

The function `prettifyClause(M)` converts the set of clauses `M` into a string.

In [ ]:
function prettify(M: RS<Clause>): string {
    const clauseStrings: string[] = [];  
    for (const clause of M) {
        clauseStrings.push(prettifyClause(clause));
    }
    return `{${clauseStrings.join(', ')}}`;
}

Given two clauses $C_1$ and $C_2$ that are represented as sets of literals, the function `cutRule`$(C_1, C_2)$ computes the set of all clauses that can be derived from $C_1$ and $C_2$ using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$

In [ ]:
function cutRule(C1: Clause, C2: Clause): RS<Clause> {
    return C1.reduce((r, l) => {
        const lBar = complement(l);
        if (C2.has(lBar)) {
            // (C1 \ {l}) U (C2 \ {lBar})
            r.add(C1.difference(single(l)).union(C2.difference(single(lBar))));
        }
        return r;
    }, empty<Clause>()); 
}

In [ ]:
const C1: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'q'), new Tuple('¬', 'r'));
const C2: Clause = new RecursiveSet<Literal>('q', 'r');
prettify(cutRule(C1, C2));

In the expression `saturate(Clauses)` below, `Clauses` is a set of *clauses*, where each clause is a set of *literals*.  The call `saturate(Clauses)` computes the set of all clauses that can be derived from clauses in the set `Clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the set `Clauses` and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D \in \texttt{Clauses}$ or $\{\} \in \texttt{Clauses}$.

In [ ]:
function saturate(Clauses: RS<Clause>): RS<Clause> {
    let currentClauses = Clauses.clone(); 
    while (true) {
        const sizeBefore = currentClauses.size;
        let newClauses = empty<Clause>();
        for (const C1 of currentClauses) {
            for (const C2 of currentClauses) {
                const derived = cutRule(C1, C2)
                if (derived.some(D => D.isEmpty())) {
                    return single(empty<Literal>());
                }
                newClauses = newClauses.union(derived);
            }
        }
        currentClauses = currentClauses.union(newClauses);
        if (currentClauses.size === sizeBefore) {
            return currentClauses;
        }
    }
}

In [ ]:
const C1: Clause = new RecursiveSet<Literal>('p', 'q');
const C2: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'));
const C3: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), new Tuple('¬', 'q'));
const Clauses: RecursiveSet<Clause> = new RecursiveSet<Clause>(C1, C2, C3);
prettify(saturate(Clauses));

The function $\texttt{findValuation}(\texttt{Clauses})$ takes a set of clauses as input.  The function tries to compute a variable interpretation that makes all of these clauses `true`.  If this attempt is successful, a set of literals is returned.  This set of literals does not contain  any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If $\texttt{Clauses}$ is unsatisfiable, `false` is returned.

In [ ]:
function restFalse(C: Clause, p: Literal, Complements: RecursiveSet<Literal>): boolean {
    return [...C].every(l => l == p || Complements.has(l));
}

In [ ]:
function findValuation(Clauses: RS<Clause>): false | RS<Literal> {
    const saturated = saturate(Clauses);
    if (saturated.some(c => c.isEmpty())) {
        return false;
    }        
    const Variables = collectVariables(Clauses);
    return Variables.reduce((Literals, p) => {
        const Complements = Literals.map(complement);         
        if (saturated.some(C => C.has(p) && restFalse(C, p, Complements))) {
            Literals.add(p);
        } else {
            Literals.add(complement(p));
        }        
        return Literals;
    }, empty<Literal>()); 
}

In [ ]:
const C01: Clause = new RecursiveSet<Literal>('r', 'p', 's');
const C02: Clause = new RecursiveSet<Literal>('r', 's');
const C03: Clause = new RecursiveSet<Literal>('p', 'q', 's');
const C04: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), new Tuple('¬', 'q'));
const C05: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), 's', new Tuple('¬', 'r'));
const C06: Clause = new RecursiveSet<Literal>('p', new Tuple('¬', 'q'), 'r');
const C07: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'r'), new Tuple('¬', 's'), 'q');
const C08: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), new Tuple('¬', 's'));
const C09: Clause = new RecursiveSet<Literal>('p', new Tuple('¬', 'r'), new Tuple('¬', 'q'));
const C10: Clause = new RecursiveSet<Literal>(new Tuple('¬', 'p'), 'r', 'q', new Tuple('¬', 's'));
const C11: Clause = new RecursiveSet<Literal>('p', 'q', 'r', new Tuple('¬', 's'));

const Clauses1: RecursiveSet<Clause> = new RecursiveSet<Clause>(
  C01, C02, C03, C04, C05, C06, C07, C08, C09, C10
);

const valuation = findValuation(Clauses1);
if (valuation) {
    console.log(prettifyClause(valuation));
} else {
    console.log("false");
}

In [ ]:
const Clauses2: RecursiveSet<Clause> = new RecursiveSet<Clause>(
  C01, C02, C03, C04, C05, C06, C07, C08, C09, C10, C11
);
const valuation = findValuation(Clauses2);
if(valuation){
    console.log(prettifyClause(valuation));
} else{
    console.log("false")
}

In [ ]:
saturate(Clauses2);